# Brazilian E-Commerce (Olist) — Exploratory Data Analysis
**Dataset**: Olist Brazilian E-Commerce Public Dataset (Kaggle)<br>
**Time Span**: Sep 2016 — Oct 2018<br>
**Core Tables**: `orders`, `customers`, `order_items`, `products`, `category_translation`<br>
**Analysis Goal**: Understand customer behaviour, order patterns, and regional characteristics to support marketing and operational decisions.<br><br>This notebook serves as the **foundational EDA** for the deeper analyses in:<br>
- `p2_cohort_analysis` — Customer retention<br>
- `p3_rfm_segmentation` — Customer value segmentation<br>
- `p4_product_location` — Product & regional insights

## 1. Environment Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
# Set consistent visual style
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150
# Use relative path — assumes notebook is in /notebooks/ and db is in /data/
DB_PATH = Path('data/ecommerce.db').resolve()
conn = sqlite3.connect(DB_PATH)
# Load core tables
orders = pd.read_sql_query('SELECT * FROM orders', conn)
customers = pd.read_sql_query('SELECT * FROM customers', conn)
order_items = pd.read_sql_query('SELECT * FROM order_items', conn)
products = pd.read_sql_query('SELECT * FROM products', conn)
category_translation = pd.read_sql_query('SELECT * FROM category_translation', conn)
print('Tables loaded successfully.')
print(f"orders: {len(orders):,} rows | customers: {len(customers):,} rows | "      f"order_items: {len(order_items):,} rows | products: {len(products):,} rows")

## 2. Data Quality Assessment
### 2.1 Table Overview & Schema

In [ ]:
# Table sizes
print(f"{'='*50}")
print("1.Table Overview\n")
table_sizes = pd.read_sql_query('''
    SELECT 'orders' AS table_name, COUNT(*) AS row_count FROM orders
    UNION ALL
    SELECT 'customers', COUNT(*) FROM customers
    UNION ALL
    SELECT 'order_items', COUNT(*) FROM order_items
    UNION ALL
    SELECT 'products', COUNT(*) FROM products
    UNION ALL
    SELECT 'category_translation', COUNT(*) FROM category_translation
''', conn)
display(table_sizes)
# Schema overview
for name, df in [('orders', orders), ('customers', customers), ('order_items', order_items), ('products', products)]:
    print(f'--- {name} ---')
    print(df.dtypes)
    print()
print(f"{'='*50}")

### 2.2 Missing Values

In [ ]:
print("2.Missing Values\n")
def missing_summary(df,table_name):
    print(f"\ntable name:{table_name}")
    null_col=df.isnull().sum()[df.isnull().sum()>0]
    if len(null_col)>0 :
        miss_df = pd.DataFrame({
            "missing_count": null_col,
            "missing_pct": round(100.0 * null_col / len(df), 2)
        })
        miss_df.columns = ["Missing Count", "Missing %"]
        display(miss_df)
    else: print("no missing values")

tables = {
    'orders': orders,
    'customers': customers,
    'order_items': order_items,
    'products': products,
}
for name, df in tables.items():
	missing_summary(df,name)
print(f"{'='*50}")

### 2.3 Duplicate & Referential Integrity Checks

In [ ]:
print("3.Duplicates Check and Referential Integrity\n")
key_checks={
	'orders':'order_id',
	'customers':'customer_unique_id',
	'products': 'product_id',
}
for table, key_col in key_checks.items():
	df = tables[table]
	dup_rate = df[key_col].duplicated().sum() / len(df)
	if dup_rate > 0:
		print(f" {table}.{key_col} Duplication Rate: {dup_rate:.2%}")
	else: 
		print(f"  {table}.{key_col}: No duplicates")
print(f"{'='*50}")

def check_relationships():
	orphan_items=set(order_items['order_id']) - set(orders['order_id'])
	print(f"order_items.order_id not in orders: {len(orphan_items)}")
	orphan_cus=set(orders['customer_id']) - set(customers['customer_id'])
	print(f"orders.customer_id not in customers: {len(orphan_cus)}")
	orphan_products = set(order_items['product_id']) - set(products['product_id'])
	print(f"order_items.product_id not in products: {len(orphan_products)}")
	print(f"order_status values: {orders['order_status'].unique()}")

check_relationships()
print(f"{'='*50}")

### 2.4 Outlier & Logic Checks

In [ ]:
print("4.Outlier Check\n")
def check_outlier():
	#1.日期倒挂 Negative delivery time（发货时间早于下单时间）
	orders_temp=orders.copy()
	orders_temp['delivered']=pd.to_datetime(orders_temp['order_delivered_carrier_date'])
	orders_temp['purchase']=pd.to_datetime(orders_temp['order_purchase_timestamp'])
	time=orders_temp[orders_temp['delivered']<orders_temp['purchase']]
	print(f"  Negative delivery time : {len(time)}")
	#2.金额为负或0 Non-positive price
	price=order_items[order_items['price']<=0]
	print(f"  Non-positive price: {len(price)}")
	#3.极端值 Extreme price
	q99=order_items['price'].quantile(0.99)
	extreme=order_items[order_items['price']>q99*10]
	print(f"  Extreme high price (>10x P99): {len(extreme)}")

check_outlier()
print(f"{'='*50}")

### 2.5 Time Range

In [ ]:
print("5.Time Range\n")
time_range = pd.read_sql_query('''
    SELECT
        MIN(order_purchase_timestamp) AS earliest,
        MAX(order_purchase_timestamp) AS latest,
        COUNT(DISTINCT DATE(order_purchase_timestamp)) AS unique_days
    FROM orders
    WHERE order_status = 'delivered'
''', conn)
display(time_range)

## 3. Order-Level Analysis<br>
### 3.1 Order Status Distribution*Covers: Q04*

In [ ]:
status_dist = pd.read_sql_query('''
    SELECT order_status, COUNT(*) AS cnt
    FROM orders
    GROUP BY order_status
    ORDER BY cnt DESC
''', conn)
status_dist['pct'] = (status_dist.cnt / status_dist.cnt.sum() * 100).round(2)
display(status_dist)
# Visualise
fig, ax = plt.subplots(figsize=(8, 4))
colors = sns.color_palette('viridis', len(status_dist))
ax.barh(status_dist.order_status, status_dist.cnt, color=colors)
ax.set_xlabel('Order Count')
ax.set_title('Order Status Distribution')
for i, (cnt, pct) in enumerate(zip(status_dist.cnt, status_dist.pct)):
    ax.text(cnt + 500, i, f'{cnt:,} ({pct}%)', va='center', fontsize=9)
plt.tight_layout()
plt.show()

> **Key Finding**:`delivered` dominates the dataset. For downstream analyses (Cohort, RFM, Product), we filter to `delivered` orders to ensure data quality and business relevance.
 ### 3.2 Order Value Statistics

In [ ]:
order_values = pd.read_sql_query('''
    SELECT o.order_id, SUM(oi.price) AS order_total
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY o.order_id
''', conn)
print(order_values.order_total.describe())
print()
for p in [10, 25, 50, 75, 90, 95, 99]:
    print(f'P{p}: {order_values.order_total.quantile(p/100):.2f}')

### 3.3 Order Value Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
# Histogram
axes[0].hist(order_values.order_total, bins=100, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_xlim(0, 1000)
axes[0].set_xlabel('Order Total (BRL)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Order Value Distribution (capped at 1,000 BRL)')
# Boxplot
axes[1].boxplot(order_values.order_total, vert=False)
axes[1].set_xlabel('Order Total (BRL)')
axes[1].set_title('Order Value Boxplot')
plt.tight_layout()
plt.show()

### 3.4 Items per Order Distribution*Covers: Q16*

In [ ]:
items_per_order = pd.read_sql_query('''
    SELECT o.order_id, COUNT(oi.product_id) AS item_count, SUM(oi.price) AS order_total
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY o.order_id
''', conn)
item_summary = items_per_order.groupby('item_count').agg(
    order_cnt=('order_id', 'count'),
    avg_value=('order_total', 'mean')
).reset_index()
display(item_summary.head(10))
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(item_summary.item_count.head(10), item_summary.order_cnt.head(10), color='coral')
ax.set_xlabel('Items per Order')
ax.set_ylabel('Number of Orders')
ax.set_title('Distribution of Items per Order')
plt.tight_layout()
plt.show()

## 4. Customer-Level Analysis
### 4.1 Customer Geographic Distribution

In [ ]:
geo_dist = pd.read_sql_query('''
    SELECT c.customer_state,
        COUNT(DISTINCT c.customer_unique_id) AS unique_customers,
        COUNT(o.order_id) AS total_orders
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_state
    ORDER BY unique_customers DESC
''', conn)
geo_dist['orders_per_customer'] = (geo_dist.total_orders / geo_dist.unique_customers).round(2)
display(geo_dist.head(10))
fig, ax = plt.subplots(figsize=(10, 5))
top10 = geo_dist.head(10)
ax.barh(top10.customer_state[::-1], top10.unique_customers[::-1], color='teal')
ax.set_xlabel('Unique Customers')
ax.set_title('Top 10 States by Customer Count')
plt.tight_layout()
plt.show()

### 4.2 Customer Lifespan Distribution

In [ ]:
lifespan_df = pd.read_sql_query('''
    SELECT c.customer_unique_id,
        MIN(o.order_purchase_timestamp) AS first_order,
        MAX(o.order_purchase_timestamp) AS last_order
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_unique_id''', conn)
lifespan_df['first_order'] = pd.to_datetime(lifespan_df.first_order)
lifespan_df['last_order'] = pd.to_datetime(lifespan_df.last_order)
lifespan_df['lifespan_days'] = (lifespan_df.last_order - lifespan_df.first_order).dt.days
bins = [-1, 0, 30, 90, 180, 99999]
labels = ['0 days', '1-30 days', '31-90 days', '91-180 days', '180+ days']
lifespan_df['lifespan_group'] = pd.cut(lifespan_df.lifespan_days, bins=bins, labels=labels)
lifespan_summary = lifespan_df.groupby('lifespan_group', observed=True).size().reset_index(name='customer_count')
lifespan_summary['pct'] = (lifespan_summary.customer_count / lifespan_summary.customer_count.sum() * 100).round(2)
display(lifespan_summary)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(lifespan_summary.lifespan_group.astype(str), lifespan_summary.customer_count, color='mediumpurple')
ax.set_xlabel('Customer Lifespan')
ax.set_ylabel('Customer Count')
ax.set_title('Customer Lifespan Distribution')
for i, (cnt, pct) in enumerate(zip(lifespan_summary.customer_count, lifespan_summary.pct)):
    ax.text(i, cnt + 200, f'{pct}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

> **Key Finding**: The vast majority of customers have a lifespan of 0 days (single purchase), confirming the one-time buyer dominance. This directly motivates the **Cohort Analysis (P2)** and **RFM Segmentation (P3)**.
### 4.3 Time to 2nd Purchase

In [ ]:
repurchase = pd.read_sql_query('''
    WITH ranked AS (
        SELECT c.customer_unique_id,
            o.order_purchase_timestamp,
            ROW_NUMBER() OVER (PARTITION BY c.customer_unique_id
                                ORDER BY o.order_purchase_timestamp) AS rn
        FROM orders o
        JOIN customers c ON o.customer_id = c.customer_id
        WHERE o.order_status = 'delivered'    )

    SELECT ROUND(AVG(JULIANDAY(r2.order_purchase_timestamp) - 
        JULIANDAY(r1.order_purchase_timestamp)), 1) AS avg_days_to_2nd
    FROM ranked r1
    JOIN ranked r2 ON r1.customer_unique_id = r2.customer_unique_id
    WHERE r1.rn = 1 AND r2.rn = 2
''', conn)
display(repurchase)

> **Key Finding**: Average time to 2nd purchase is ~81 days. This long gap suggests a low-engagement market, reinforcing the need for retention strategies.
## 5. Time-Series Analysis
### 5.1 Monthly Order Trend

In [ ]:
monthly = pd.read_sql_query('''
    SELECT
        STRFTIME('%Y-%m', order_purchase_timestamp) AS month,
        COUNT(*) AS order_count
    FROM orders
    WHERE order_status = 'delivered'
    GROUP BY month
    ORDER BY month
''', conn)
monthly['month_dt'] = pd.to_datetime(monthly.month)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(monthly.month_dt, monthly.order_count, marker='o', color='darkgreen')
ax.set_xlabel('Month')
ax.set_ylabel('Order Count')
ax.set_title('Monthly Order Trend (Delivered Orders)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5.2 Hour-of-Day Pattern

In [ ]:
hourly = pd.read_sql_query('''
    SELECT
        CAST(STRFTIME('%H', order_purchase_timestamp) AS INTEGER) AS hour,
        COUNT(*) AS order_count
    FROM orders
    WHERE order_status = 'delivered'
    GROUP BY hour
    ORDER BY hour
''', conn)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(hourly.hour, hourly.order_count, marker='o', color='navy')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Order Count')
ax.set_title('Orders by Hour of Day')
ax.set_xticks(range(0, 24))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5.3 Day-of-Week Pattern

In [ ]:
dow = pd.read_sql_query('''
    SELECT
        CASE CAST(STRFTIME('%w', order_purchase_timestamp) AS INTEGER) 
            WHEN 0 THEN 'Sun' WHEN 1 THEN 'Mon' WHEN 2 THEN 'Tue'
            WHEN 3 THEN 'Wed' WHEN 4 THEN 'Thu' WHEN 5 THEN 'Fri' WHEN 6 THEN 'Sat'
            END AS weekday,
        COUNT(*) AS order_count
    FROM orders
    WHERE order_status = 'delivered'
    GROUP BY weekday
''', conn)
day_order = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
dow['weekday'] = pd.Categorical(dow.weekday, categories=day_order, ordered=True)
dow = dow.sort_values('weekday')
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(dow.weekday.astype(str), dow.order_count, color='indianred')
ax.set_xlabel('Day of Week')
ax.set_ylabel('Order Count')
ax.set_title('Orders by Day of Week')
plt.tight_layout()
plt.show()

## 6. Product & Regional Analysis
### 6.1 Top Product Categories

In [ ]:
categories = pd.read_sql_query('''
    SELECT ct.product_category_name_english AS category,
        COUNT(DISTINCT o.order_id) AS order_count,
        ROUND(SUM(oi.price), 2) AS revenue
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products p ON oi.product_id = p.product_id
    JOIN category_translation ct ON p.product_category_name = ct.product_category_name
    WHERE o.order_status = 'delivered'
    GROUP BY category
    ORDER BY order_count DESC
    LIMIT 15
''', conn)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(categories.category[::-1], categories.order_count[::-1], color='goldenrod')
ax.set_xlabel('Order Count')
ax.set_ylabel('Product Category')
ax.set_title('Top 15 Product Categories by Order Volume')
plt.tight_layout()
plt.show()

### 6.2 State-Level AOV Comparison

In [ ]:
state_aov = pd.read_sql_query('''
    SELECT c.customer_state AS state,
        COUNT(DISTINCT o.order_id) AS order_count,
        ROUND(SUM(oi.price), 2) AS revenue,
        ROUND(SUM(oi.price) / COUNT(DISTINCT o.order_id), 2) AS aov
    FROM orders o
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
    GROUP BY state
    HAVING order_count > 100
    ORDER BY aov DESC
''', conn)
display(state_aov.head(10))
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(state_aov.state[::-1], state_aov.aov[::-1], color='seagreen')
ax.set_xlabel('Average Order Value (BRL)')
ax.set_title('AOV by State (Orders > 100)')
plt.tight_layout()
plt.show()

### 6.3 Delivery Time by State

In [ ]:
delivery = pd.read_sql_query('''
    SELECT c.customer_state AS state,
        COUNT(o.order_id) AS order_count,
        ROUND(AVG(CAST(JULIANDAY(o.order_delivered_customer_date) AS INTEGER) - 
            CAST(JULIANDAY(o.order_purchase_timestamp) AS INTEGER)), 1) AS avg_delivery_days
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
        AND o.order_delivered_customer_date IS NOT NULL
        AND o.order_purchase_timestamp IS NOT NULL
    GROUP BY state
    HAVING order_count > 100
    ORDER BY avg_delivery_days DESC
''', conn)
display(delivery)
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(delivery.state[::-1], delivery.avg_delivery_days[::-1], color='salmon')
ax.set_xlabel('Average Delivery Days')
ax.set_title('Delivery Time by State')
plt.tight_layout()
plt.show()

## 7. Key Findings Summary<br>
| Finding | Implication | Downstream Analysis |
|---------|-------------|---------------------|
| **97%+ customers are one-time buyers** | Extremely low retention; acquisition-heavy business model | Cohort Analysis (P2) to diagnose when/why customers drop off |
| **Average time to 2nd purchase: ~81 days** | Long re-engagement window; email/SMS campaigns should target Day 30-60 | RFM Segmentation (P3) to identify who is worth re-engaging |
| **Orders peak in afternoon hours (14:00-16:00)** | Optimal time for push notifications and flash sales | — |
| **SP (São Paulo) dominates customer base (~42%)** | Geographic concentration risk; expansion opportunity in underserved states | Product-Location Analysis (P4) |
| **AOV varies 2x across states** | Pricing and assortment strategy should be region-specific | Product-Location Analysis (P4) |
| **Delivery time ranges from 7 to 28 days by state** | Logistics is a key operational lever; long delivery correlates with lower repeat rates | — |
| **Health & Beauty, Watches & Gifts are top categories** | Core revenue drivers; cross-sell potential within these categories | Product-Location Analysis (P4) |

**Next Steps**:<br>
- **P2 — Cohort Analysis**: Deep-dive into monthly retention curves and identify high-quality acquisition cohorts.<br>
- **P3 — RFM Segmentation**: Segment customers by Recency, Frequency, and Monetary value to prioritise marketing spend.<br>
- **P4 — Product & Location Insights**: Analyse category preferences by state and build a strategic product matrix.

In [ ]:
conn.close()
print('EDA complete. Connection closed.')